In [43]:
import os

out_dir = "../data/processed"
os.makedirs(out_dir, exist_ok=True)

print("Created/exists:", os.path.abspath(out_dir))
print("Writable:", os.access(out_dir, os.W_OK))

Created/exists: /Users/malki/Documents/Skole/Masteroppgave/Master_snow_models/data/processed
Writable: True


In [44]:
import geopandas as gpd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from shapely.vectorized import contains

In [45]:
fsm2 = xr.open_dataset("../data/FSM2/fsm2_sd_2023.nc")
sen  = xr.open_dataset("../data/seNorge/sd_2023.nc")

In [46]:
regions = gpd.read_file("../data/GIS/forecasting_regions_wgs84.geojson")
jotun = regions[regions["name"] == "Jotunheimen"]

poly = jotun.geometry.values[0]

In [47]:
lat = fsm2["lat"].values
lon = fsm2["lon"].values

mask = contains(poly, lon, lat)

/var/folders/4n/m1gxpf611c158s8wx2jzfqgw0000gn/T/ipykernel_79625/2583918906.py:4: DeprecationWarning: The 'shapely.vectorized.contains' function is deprecated and will be removed a future version. Use 'shapely.contains_xy' instead (available since shapely 2.0.0).
  mask = contains(poly, lon, lat)


In [48]:
mask_xr = xr.DataArray(
    mask,
    dims=("y","x"),
    coords={"y": fsm2.y, "x": fsm2.x}
)

fsm2_jotun = fsm2.where(mask_xr)

In [49]:
sen_sd = sen["snow_depth"]

# convert cm → mm
sen_sd_mm = sen_sd * 10

In [50]:
sen_on_fsm_all = sen_sd_mm.interp(
    x=fsm2_jotun["x"],
    y=fsm2_jotun["y"],
    method="nearest"
)

In [51]:
def get_winter(data, start_year):

    start = f"{start_year}-10-01"
    end   = f"{start_year+1}-06-30"

    return data.sel(time=slice(start, end))

In [52]:
fsm_winter = get_winter(fsm2_jotun["snow_depth"], 2022)
sen_winter = get_winter(sen_on_fsm_all, 2022)

In [53]:
diff = fsm_winter - sen_winter

In [54]:
fsm2_jotun.to_netcdf("../data/processed/fsm2_jotun_2023.nc")